# 07 — Spark Structured Streaming: Kafka → Parquet

## Zweck
**Spark liest die Open-Meteo-Events aus dem Kafka-Topic** (Pflichtanforderung), validiert sie gegen
den Event-Vertrag aus Notebook `06`, reichert sie mit der Stadtreferenz an und schreibt das Ergebnis
als Parquet. Damit ist der Weg Kafka → Spark → Speicher nachgewiesen. Das Kafka-Topic ist hier die
Rohschicht; Spark erzeugt daraus die Silver-Tabelle.

## Vorgehen (wie in der Vorlesung, class6)
`spark.readStream.format("kafka")…load()` → `from_json()` mit explizitem Schema → Validierung →
`writeStream.format("parquet")`. Mit `trigger(availableNow=True)` verarbeitet Spark alle vorhandenen
Events und stoppt dann — ideal für einen reproduzierbaren Notebook-Lauf.

## Ausgabe
- Silver: `data/silver/open_meteo_city_hourly/` (validiert und mit Stadtdaten angereichert)

## Konfiguration
`SPARK_MASTER_URL` und der Kafka-Connector kommen aus der `.env`: lokal `spark://spark-master:7077`,
auf der FH `local[*]`.

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env", override=False)
DATA_DIR = PROJECT_ROOT / "data"

SPARK_MASTER_URL = os.getenv("SPARK_MASTER_URL", "local[*]")
KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:9092")
KAFKA_TOPIC = os.getenv("KAFKA_TOPIC_AIR_QUALITY_LIVE", "air_quality_live")
CONNECTOR = os.getenv("SPARK_KAFKA_CONNECTOR_PACKAGE", "")

SILVER_STREAM = DATA_DIR / "silver" / "open_meteo_city_hourly"
CHECKPOINT = DATA_DIR / "checkpoints" / "open_meteo_stream"
print({"spark_master": SPARK_MASTER_URL, "kafka": KAFKA_BOOTSTRAP_SERVERS, "topic": KAFKA_TOPIC})

{'spark_master': 'spark://spark-master:7077', 'kafka': 'kafka:29092', 'topic': 'air_quality_live'}


## Sauberer Neulauf
Wir lesen ohnehin alle Events ab dem frühesten Offset (`startingOffsets=earliest`). Damit jeder Lauf
reproduzierbar von vorne beginnt, entfernen wir vorab die alte Ausgabe und den Checkpoint. Sonst würde
Spark aus einem alten Checkpoint fortsetzen (z. B. mit einem nicht mehr existierenden Topic).

In [2]:
import shutil

shutil.rmtree(SILVER_STREAM, ignore_errors=True)
shutil.rmtree(CHECKPOINT, ignore_errors=True)
print("Alte Ausgabe und Checkpoint entfernt.")

Alte Ausgabe und Checkpoint entfernt.


## Spark-Session starten

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

builder = SparkSession.builder.appName("open-meteo-kafka-to-parquet").master(SPARK_MASTER_URL)
if CONNECTOR:
    builder = builder.config("spark.jars.packages", CONNECTOR)
spark = builder.config("spark.sql.shuffle.partitions", "1").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print({"spark_version": spark.version, "master": spark.sparkContext.master})

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8671710b-e951-48d5-b94f-1cc7c940b180;1.0
	confs: [default]


	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.7 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.7 in central


	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central


	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central


:: resolution report :: resolve 882ms :: artifacts dl 52ms
	:: modules in use:
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	commons-logging#commons-logging;1.1.3 from central in [default]
	org.apache.commons#commons-pool2;2.11.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.3.4 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.3.4 from central in [default]
	org.apache.kafka#kafka-clients;3.4.1 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.7 from central in [default]
	org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.7 from central in [default]
	org.lz4#lz4-java;1.8.0 from central in [default]
	org.slf4j#slf4j-api;2.0.7 from central in [default]
	org.xerial.snappy#snappy-java;1.1.10.5 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded

26/06/29 18:07:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


{'spark_version': '3.5.7', 'master': 'spark://spark-master:7077'}


## Kafka-Stream lesen und JSON parsen
Das `value`-Feld kommt als Bytes; wir casten zu String und parsen mit einem **expliziten** Schema
(kein Schema-Inferenz im Streaming).

In [4]:
event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("schema_version", StringType()),
    StructField("source", StringType()),
    StructField("city_id", StringType()),
    StructField("event_time_utc", StringType()),
    StructField("pm2_5", DoubleType()),
    StructField("pm10", DoubleType()),
    StructField("no2", DoubleType()),
])

kafka_raw = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)
parsed = (
    kafka_raw.selectExpr("CAST(value AS STRING) AS json")
    .select(from_json("json", event_schema).alias("e"))
    .select("e.*")
)
print("Kafka-Stream definiert und JSON-Schema angewendet.")

Kafka-Stream definiert und JSON-Schema angewendet.


## Validierung und Anreicherung
Wir behalten nur Events, die dem Vertrag entsprechen (richtige `schema_version`/`source`, Pflichtfelder
vorhanden, Schadstoffwerte in plausiblen Bereichen), entfernen Duplikate über `event_id` und verknüpfen
per Join mit der Stadtreferenz. Unbekannte `city_id` fallen durch den Inner-Join heraus.

In [5]:
valid = parsed.filter(
    (col("schema_version") == "1.0")
    & (col("source") == "open_meteo")
    & col("event_id").isNotNull()
    & col("city_id").isNotNull()
    & col("event_time_utc").isNotNull()
    & col("pm2_5").between(0, 1000)
    & col("pm10").between(0, 2000)
    & col("no2").between(0, 1000)
).dropDuplicates(["event_id"])

city_reference = spark.read.parquet(str(DATA_DIR / "silver" / "city_reference.parquet")) \
    .select("city_id", "city_name")

enriched = (
    valid.join(city_reference, on="city_id", how="inner")
    .withColumn("event_time_ts", to_timestamp("event_time_utc"))
    .select("event_id", "city_id", "city_name", "event_time_ts", "pm2_5", "pm10", "no2")
)
print("Validierung und Anreicherung definiert.")

Validierung und Anreicherung definiert.


## Silver schreiben und Nachweis: Spark hat aus Kafka gelesen

In [ ]:
try:
    query = (
        enriched.writeStream.format("parquet")
        .option("path", str(SILVER_STREAM))
        .option("checkpointLocation", str(CHECKPOINT))
        .trigger(availableNow=True)
        .start()
    )
    query.awaitTermination()
    print(f"Silver-Stream geschrieben: {SILVER_STREAM}")

    silver_count = spark.read.parquet(str(SILVER_STREAM)).count()
    spark_read_kafka_requirement_proven = silver_count > 0
    print({"silver_zeilen": silver_count,
           "spark_read_kafka_requirement_proven": spark_read_kafka_requirement_proven})
    assert spark_read_kafka_requirement_proven, "Es wurden keine Events aus Kafka verarbeitet."
    spark.read.parquet(str(SILVER_STREAM)).show(5, truncate=False)
finally:
    # Session immer schliessen, auch wenn der Assert oben fehlschlaegt - sonst bleibt die JVM
    # mit dem alten Master/Connector haengen und ein Neulauf im selben Kernel wiederholt nur
    # denselben Fehler (Restart noetig, um SPARK_MASTER_URL/SPARK_KAFKA_CONNECTOR_PACKAGE neu
    # zu laden, da SparkSession.getOrCreate() eine bestehende Session sonst weiterverwendet).
    spark.stop()

## Nächster Schritt
Notebook `08` ausführen — die Gold-Tabellen und den Qualitätsbericht erzeugen.